In [ ]:
import numpy as np
import sys
import os
from autokmc.structure import build_surface, build_nanoparticle
from ase.visualize.x3d import view_x3d
from autokmc.surface import find_surface_atoms
import copy
from collections import Counter
from autokmc.graph import build_graph
from autokmc.site import find_adsorption_sites

In [ ]:
# Ensure the project root is on the path when running from the autokmc/ subdirectory
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
### Load the Allegro/NequIP calculator
import torch
from nequip.ase import NequIPCalculator

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_MODEL_FILE = "asehcocuau.nequip.pt2" if _DEVICE == "cuda" else "cpuhcocuau.nequip.pth"
_MODEL_PATH = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", _MODEL_FILE)
print(f"Using device : {_DEVICE}")
print(f"Model file   : {_MODEL_FILE}")

def make_calc():
    """Return a fresh NequIPCalculator instance loaded from the model."""
    return NequIPCalculator.from_compiled_model(
        compile_path=_MODEL_PATH,
        device=_DEVICE,
    )

calc = make_calc()
print(f"Calculator : {calc.__class__.__name__}")
print(f"Model      : {_MODEL_PATH}")

In [ ]:
## Build a Cu(111) surface slab
slab = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(1, 1, 1),
    calculator=make_calc(),
    min_slab_size=8.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True
)

print(f"\nSlab formula : {slab.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab)}")
cell = slab.get_cell()
print(f"Cell (Å)     : a={cell[0,0]:.3f}  b={cell[1,1]:.3f}  c={cell[2,2]:.3f}")

In [ ]:
## Visualise the slab
view_x3d(slab)

In [ ]:
### Get surface atoms

surface_mask, surface_indices, method = find_surface_atoms(
    slab,
    which="top",
    tag_atoms=True,   # writes slab.arrays["surface"] for extxyz export
)

print(f"Detection method : {method}")
print(f"Surface atoms    : {surface_mask.sum()} / {len(slab)}")
print(f"Surface indices  : {surface_indices}")

In [ ]:
### Visualise surface atoms in X3D
# Surface atoms are shown as Au (gold), bulk atoms remain as Cu
# so the two populations are visually distinct in x3d.
slab_vis = copy.deepcopy(slab)
symbols = np.array(slab_vis.get_chemical_symbols())
symbols[surface_indices] = "Au"
slab_vis.set_chemical_symbols(symbols.tolist())

view_x3d(slab_vis)

In [ ]:
### Build the graph for the slab
graph = build_graph(slab)
print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
# Show breakdown by node type
type_counts = Counter(d["type"] for _, d in graph.nodes(data=True))
for t, n in sorted(type_counts.items()):
    print(f"  {t:10s} : {n}")

In [ ]:
### Build reactants
from autokmc.reactants import build_reactant

c    = build_reactant("[C]", calculator=make_calc())
co   = build_reactant("[C-]#[O+]", calculator=make_calc())
o2   = build_reactant("O=O",       calculator=make_calc())
ch4  = build_reactant("C",         calculator=make_calc())
ch3  = build_reactant("[CH3]",     calculator=make_calc())
ch2  = build_reactant("[CH2]",     calculator=make_calc())
ch   = build_reactant("[CH]",      calculator=make_calc())
o    = build_reactant("[O]",       calculator=make_calc())

for r in [o, co, o2, ch4, ch3, ch2, ch]:
    print(f"\nReactant : {r.smiles}")
    print(f"  Formula  : {r.atoms.get_chemical_formula()}")
    print(f"  Atoms    : {len(r.atoms)}")
    print(f"  Nodes    : {r.graph.number_of_nodes()}")
    print(f"  Edges    : {r.graph.number_of_edges()}")
    for i, d in r.graph.nodes(data=True):
        pos = d['position']
        print(f"    node {i}  element={d['element']:2s}  type={d['type']}  r_cov={d['covalent_radius']:.3f} Å  pos=({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) Å")

In [ ]:
### Site visualisation helper
# Each iso-class gets a distinct proxy element so they appear in different colours.
_ISO_PROXY = ["Au", "Pt", "Pd", "Ag", "Ir", "Rh", "Os", "Re"]

def _make_site_vis(slab, sites, reactant):
    """Return an Atoms object: slab + one representative per iso-class."""
    import copy as _copy
    vis = _copy.deepcopy(slab)
    seen_classes = set()
    for s in sites:
        if s.iso_class in seen_classes:
            continue
        seen_classes.add(s.iso_class)
        proxy = _ISO_PROXY[s.iso_class % len(_ISO_PROXY)]
        if len(reactant.atoms) == 1:
            vis.append(proxy)
            vis.positions[-1] = s.position
        else:
            ads_syms = reactant.atoms.get_chemical_symbols()
            for k, sym in enumerate(ads_syms):
                vis.append(proxy if k == 0 else sym)
                vis.positions[-1] = s.position[k]
    return vis

def _print_sites_single(label, sites):
    print(f"\n{label} sites summary")
    print(f"  Total sites  : {len(sites)}")
    for iso_cid, count in sorted(Counter(s.iso_class for s in sites).items()):
        rep = next(s for s in sites if s.iso_class == iso_cid)
        e_str = f"  E_ads={rep.energy:.4f} eV  converged={rep.converged}" if rep.energy is not None else ""
        print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites{e_str}  "
              f"pos = ({rep.position[0]:.3f}, {rep.position[1]:.3f}, {rep.position[2]:.3f}) Å")

def _print_sites_multi(label, sites):
    print(f"\n{label} sites summary")
    print(f"  Total sites  : {len(sites)}")
    for iso_cid, count in sorted(Counter(s.iso_class for s in sites).items()):
        rep = next(s for s in sites if s.iso_class == iso_cid)
        n_surf = len({sg for _, sg in rep.conn_global})
        e_str = f"  E_ads={rep.energy:.4f} eV  converged={rep.converged}" if rep.energy is not None else ""
        print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites  "
              f"{n_surf} surface atoms bonded{e_str}")

In [ ]:
### Find adsorption sites — O (single atom, k_max=4)

sites_c, site_graph_c = find_adsorption_sites(
    graph, c,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_single("C", sites_c)

In [ ]:
slab_c_vis = _make_site_vis(slab, sites_c, c)
print(f"C site visualisation: {len(slab_c_vis)} atoms "
      f"({len(slab)} slab + {len(slab_c_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_c}):
    rep = next(s for s in sites_c if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_c_vis)

In [ ]:
### Find adsorption sites — O (single atom, k_max=4)

sites_o, site_graph_o = find_adsorption_sites(
    graph, o,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_single("O", sites_o)

In [ ]:
slab_o_vis = _make_site_vis(slab, sites_o, o)
print(f"O site visualisation: {len(slab_o_vis)} atoms "
      f"({len(slab)} slab + {len(slab_o_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_o}):
    rep = next(s for s in sites_o if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_o_vis)

In [ ]:
### Find adsorption sites — CO (multi-atom, k_max=3)

sites_co, site_graph_co = find_adsorption_sites(
    graph, co,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CO", sites_co)

In [ ]:
slab_co_vis = _make_site_vis(slab, sites_co, co)
print(f"CO site visualisation: {len(slab_co_vis)} atoms "
      f"({len(slab)} slab + {len(slab_co_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_co}):
    rep = next(s for s in sites_co if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_co_vis)

In [ ]:
### Find adsorption sites — O2 (multi-atom, k_max=4)

sites_o2, site_graph_o2 = find_adsorption_sites(
    graph, o2,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("O2", sites_o2)

In [ ]:
slab_o2_vis = _make_site_vis(slab, sites_o2, o2)
print(f"O2 site visualisation: {len(slab_o2_vis)} atoms "
      f"({len(slab)} slab + {len(slab_o2_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_o2}):
    rep = next(s for s in sites_o2 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_o2_vis)

In [ ]:
### Find adsorption sites — CH (multi-atom, k_max=4)

sites_ch, site_graph_ch = find_adsorption_sites(
    graph, ch,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH", sites_ch)

In [ ]:
slab_ch_vis = _make_site_vis(slab, sites_ch, ch)
print(f"CH site visualisation: {len(slab_ch_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch}):
    rep = next(s for s in sites_ch if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch_vis)

In [ ]:
### Find adsorption sites — CH2 (multi-atom, k_max=4)

sites_ch2, site_graph_ch2 = find_adsorption_sites(
    graph, ch2,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH2", sites_ch2)

In [ ]:
slab_ch2_vis = _make_site_vis(slab, sites_ch2, ch2)
print(f"CH2 site visualisation: {len(slab_ch2_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch2_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch2}):
    rep = next(s for s in sites_ch2 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch2_vis)

In [ ]:
### Find adsorption sites — CH3 (multi-atom, k_max=4)

sites_ch3, site_graph_ch3 = find_adsorption_sites(
    graph, ch3,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH3", sites_ch3)

In [ ]:
slab_ch3_vis = _make_site_vis(slab, sites_ch3, ch3)
print(f"CH3 site visualisation: {len(slab_ch3_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch3_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch3}):
    rep = next(s for s in sites_ch3 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch3_vis)

In [ ]:
### Find adsorption sites — CH4 (multi-atom, k_max=4)

sites_ch4, site_graph_ch4 = find_adsorption_sites(
    graph, ch4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH4", sites_ch4)

In [ ]:
slab_ch4_vis = _make_site_vis(slab, sites_ch4, ch4)
print(f"CH4 site visualisation: {len(slab_ch4_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch4_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch4}):
    rep = next(s for s in sites_ch4 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch4_vis)